In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from project.models import EDMEvelynn
from project.util.data import ReplayMemoryData

In [ ]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

In [ ]:
DATA = os.path.join("checkpoints", "memory", "AdamW25e5.pt")
CHECKPOINT = os.path.join("checkpoints", "diffusion", "test.pt")

In [ ]:
MODEL = "edm2"

In [ ]:
TARGET = 0
BATCH_SIZE = 64
RESOLUTION = 120
IN_CHANNELS = 4
OUT_CHANNELS = 3
START_CHANNELS = 64
NUM_RES_BLOCKS = 2
CHANNEL_MULTIPLIERS = (1, 2,)
ATTENTION_RESOLUTIONS = (8,)
LR = 2e-3
DROPOUT = 0.13

In [ ]:
transform = transforms.Compose([
    transforms.Normalize(0.5, 0.5),
])

In [ ]:
data = ReplayMemoryData(
    memory=DATA,
    transform=transform,
)
loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
image = ((data[2] + 1) / 2).permute(1, 2, 0)

plt.imshow(image)
plt.show()

In [ ]:
model = EDMEvelynn(
    img_resolution=RESOLUTION,
    img_channels=IN_CHANNELS,
    start_channels=START_CHANNELS,
    channel_mult=CHANNEL_MULTIPLIERS,
    num_blocks=NUM_RES_BLOCKS,
    attention_resolutions=ATTENTION_RESOLUTIONS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    network="edm2"
).to(device)

In [ ]:
if os.path.exists(CHECKPOINT):
    model.load_checkpoint(CHECKPOINT)
    print("Loaded checkpoint!")

In [ ]:
model.train(10, loader)

In [ ]:
model.save_checkpoint(CHECKPOINT)

In [ ]:
x = model.heun_sampler(16).to("cpu")
x = (x + 1) / 2

In [ ]:
fig, axis = plt.subplots(4, 4, figsize=(10, 10), sharex=True, sharey=True)
for i in range(4):
    for j in range(4):
        axis[i, j].imshow(x[i * 4 + j].permute(1, 2, 0).clip(0, 1))
        axis[i, j].grid(None)